In [ ]:
import json, unicodedata, torch
from typing import List, Dict, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForTokenClassification
from typing import List, Dict, Tuple, Optional
import json, unicodedata, torch
import re
import unicodedata

In [ ]:
def clean_clinical_text(text: str) -> str:
    """
    Cleans raw Portuguese clinical text before feeding into the NER model.
    Steps:
      1. Normalize Unicode (e.g., remove exotic diacritics).
      2. Remove markdown formatting (**bold**, ### headings, etc.).
      3. Replace multiple spaces / newlines with single space.
      4. Standardize punctuation and spacing.
      5. Trim leading/trailing spaces.

    Returns:
        str: Clean, normalized text.
    """

    # --- 1. Normalize Unicode ---
    text = unicodedata.normalize("NFKC", text)

    # --- 2. Remove markdown & redundant symbols ---
    text = re.sub(r"[*_#>]+", " ", text)       # remove Markdown markers
    text = re.sub(r"\s*-\s*", " - ", text)     # ensure spacing around dashes
    text = re.sub(r"\s*•\s*", " - ", text)     # bullet replacements
    text = re.sub(r"\s*\*\*\s*", " ", text)    # remove extra asterisks

    # --- 3. Remove extra spaces / newlines ---
    text = re.sub(r"\r\n|\r|\n", " ", text)    # collapse line breaks
    text = re.sub(r"\s+", " ", text)           # multiple spaces → one

    # --- 4. Fix punctuation spacing ---
    text = re.sub(r"\s*([.,;:!?()])\s*", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)           # normalize again

    # --- 5. Final trim ---
    return text.strip()

In [ ]:
# -------------------- Bases --------------------
# Bases that HAVE polarity:
TARGET_BASES_POL = ["Alergias medicamentosas"]
TARGET_POLARITIES = ["Positiva", "Negativa"]

# Bases WITHOUT polarity:
TARGET_BASES_SIMPLE = ["Diagnóstico","Medicação Habitual"]

# All supported bases:
ALL_BASES = TARGET_BASES_POL + TARGET_BASES_SIMPLE


# ---- helpers ----
def _strip_accents_lower(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().strip()

# Build canonical maps from config above
_CANON_BASE = {_strip_accents_lower(x): x for x in ALL_BASES}
_CANON_POL  = { _strip_accents_lower("Positiva"): "Positiva",
                _strip_accents_lower("Negativa"): "Negativa",
                _strip_accents_lower("Nagetiva"): "Negativa" }  # common typo

def canonical_base(raw: Optional[str]) -> Optional[str]:
    return _CANON_BASE.get(_strip_accents_lower(raw or ""), None)

def canonical_pol(raw: Optional[str]) -> Optional[str]:
    return _CANON_POL.get(_strip_accents_lower(raw or ""), None)

def decode_spans(offsets, tags: List[str]):
    """Decode BIO tag strings to (begin,end,label_str) using tokenizer offsets."""
    spans=[]; cur=None; idxs=[]
    for i,t in enumerate(tags):
        if t=="O" or "-" not in t:
            if cur and idxs:
                b=offsets[idxs[0]][0]; e=offsets[idxs[-1]][1]; spans.append((b,e,cur))
            cur=None; idxs=[]; continue
        pref, lab = t.split("-",1)
        if pref=="B":
            if cur and idxs:
                b=offsets[idxs[0]][0]; e=offsets[idxs[-1]][1]; spans.append((b,e,cur))
            cur=lab; idxs=[i]
        elif pref=="I":
            if cur==lab: idxs.append(i)
            else: cur=lab; idxs=[i]
    if cur and idxs:
        b=offsets[idxs[0]][0]; e=offsets[idxs[-1]][1]; spans.append((b,e,cur))
    return spans

def parse_base_pol(raw_lab: str):
    """From raw tag label like 'Base__Positiva' / 'Base_Negativa' / 'Diagnóstico' → (base, pol|None)."""
    if not raw_lab: return None, None
    if "__" in raw_lab: left,right = raw_lab.split("__",1)
    elif "_" in raw_lab: left,right = raw_lab.split("_",1)
    else: left,right = raw_lab, None
    base = canonical_base(left)
    pol  = canonical_pol(right) if right is not None else None
    # Force non-polarity bases to have None polarity
    if base in TARGET_BASES_SIMPLE:
        pol = None
    # Drop unknown polarities for polarity bases
    if base in TARGET_BASES_POL and (pol is not None) and (pol not in TARGET_POLARITIES):
        pol = None
    return base, pol

def ltrim_ws(text: str, b: int, e: int) -> Tuple[int,int]:
    if b is None or e is None or b<0 or e>len(text) or b>=e: return b,e
    while b<e and text[b].isspace(): b+=1
    return b,e

def _normalize_id2label(model) -> Dict[int, str]:
    """Make id2label robust to HF configs with str keys or missing maps."""
    id2label_raw = getattr(model.config, "id2label", None)
    label2id_raw = getattr(model.config, "label2id", None)
    if isinstance(id2label_raw, dict):
        try:
            return {int(k): v for k, v in id2label_raw.items()}
        except Exception:
            pass
    if isinstance(label2id_raw, dict):
        try:
            label2id = {str(k): int(v) for k, v in label2id_raw.items()}
            return {v: k for k, v in label2id.items()}
        except Exception:
            pass
    # Fallback: assume index→label ordering present in classifier
    n_labels = model.classifier.out_features if hasattr(model, "classifier") else model.num_labels
    return {i: str(i) for i in range(n_labels)}  # last resort


# ---- ONE FUNCTION ----
def glintt_finetune(text: str,
                    model_id: str = "tamunna/hfpt-glintt-balance",
                    max_len: int = 512) -> str:
    """
    Runs token classification, decodes BIO spans, and returns JSON records filtered to:
      - Bases in ALL_BASES
      - Polarity normalized to TARGET_POLARITIES, with non-polarity bases forced to None
    """
    text = clean_clinical_text(text)
    print(f"\n\n cleaned text: {text} \n\n")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to(device)
    model.eval()

    with torch.no_grad():
        enc = tok(text, return_offsets_mapping=True, truncation=True,
                  max_length=max_len, padding=False)
        offsets = enc["offset_mapping"]
        ipt = {k: torch.tensor(v).unsqueeze(0).to(device)
               for k,v in enc.items() if k!="offset_mapping"}
        logits = model(**ipt).logits.squeeze(0).cpu().numpy()
        id2label = _normalize_id2label(model)
        pred_ids = logits.argmax(-1).tolist()
        tags = ["O" if o==(0,0) else id2label[int(pid)]
                for o,pid in zip(offsets, pred_ids)]

    raw_spans = decode_spans(offsets, tags)
    records=[]
    for (b,e,raw_lab) in raw_spans:
        base, pol = parse_base_pol(raw_lab)
        if base is None or base not in ALL_BASES:
            continue
        b,e = ltrim_ws(text, b, e)
        if b<e:
            # Ensure polarity only present for polarity bases
            if base in TARGET_BASES_SIMPLE:
                pol = None
            records.append({
                "p_label": base,
                "p_polaridade": pol,  # "Positiva" | "Negativa" | None
                "p_begin": int(b),
                "p_end": int(e),
                "p_word": text[b:e]
            })
    return json.dumps(records, ensure_ascii=False, indent=2)


# **DEMO**

**In the cell below, simply modify the text that is provided as input to the model.**

In [ ]:
text1= """

### **NOTA CLÍNICA DE ADMISSÃO**

**IDENTIFICAÇÃO:**
Doente do sexo masculino, 65 anos, aposentado, natural e residente em Coimbra, casado.

**DATA E MOTIVO DA ADMISSÃO:**
Admitido em 11/02/2025 por dispneia progressiva e edema de membros inferiores.

**HISTÓRIA DA DOENÇA ATUAL:**
Paciente refere início de dispneia há aproximadamente 3 meses, inicialmente aos grandes esforços, evoluindo progressivamente para pequenos esforços e, nos últimos dias, apresentando dispneia em repouso. Relata ortopneia e episódios de dispneia paroxística noturna. Associadamente, refere edema de membros inferiores, progressivo, bilateral, depressível, com piora ao final do dia. Nega dor torácica, palpitações ou síncope.

**ANTECEDENTES PESSOAIS:**
- **Hipertensão arterial sistémica** diagnosticada há 18 anos, em tratamento com enalapril.
- **Diabetes mellitus tipo 2**, controlada com metformina.
- **Dislipidemia**, em uso de sinvastatina.
- **Doença renal crónica grau III**.
- **Ex-tabagista (30 maços/ano), cessou há 10 anos.**

**ANTECEDENTES FAMILIARES:**
- Pai falecido por insuficiência cardíaca.
- Mãe com diabetes mellitus tipo 2.

**EXAME FÍSICO:**
- **Geral:** Paciente dispneico ao repouso, ortopneico, fácies de sofrimento.
- **Sinais Vitais:**
  - **FC:** 98 bpm
  - **PA:** 140/85 mmHg
  - **FR:** 24 irpm
  - **SpO₂:** 91% em ar ambiente
  - **Temperatura:** 36,8ºC
- **Cardiovascular:** Pulso irregular, ritmo cardíaco irregularmente irregular, presença de ingurgitamento jugular, sopro sistólico ++/6+ em foco mitral, edema de membros inferiores ++/4+.
- **Respiratório:** MV reduzido globalmente, estertores crepitantes bibasais.
- **Abdómen:** Fígado palpável a 2 cm do rebordo costal.

**EXAMES COMPLEMENTARES:**
- **ECG:** Fibrilhação atrial com resposta ventricular rápida.
- **BNP:** 1.800 pg/mL.
- **Ecocardiograma:** Fração de ejeção de 35%, disfunção sistólica grave, insuficiência mitral moderada.

**LISTA DE PROBLEMAS:**
1. Insuficiência cardíaca descompensada com disfunção ventricular esquerda.
2. Fibrilhação atrial com resposta ventricular rápida.
3. Hipertensão arterial sistémica.
4. Diabetes mellitus tipo 2.
5. Dislipidemia.

**PLANO:**
- Internamento para otimização do tratamento da insuficiência cardíaca.
- Oxigenoterapia para manter SpO₂ > 92%.
- Furosemida EV para alívio da congestão.
- IECA (enalapril) e betabloqueador (carvedilol) conforme tolerância hemodinâmica.
- Considerar anticoagulação oral.
- Monitorização contínua dos sinais vitais e da função renal.
- Educação do paciente sobre a doença e medidas terapêuticas.


"""

text2= """

### **NOTA CLÍNICA DE ADMISSÃO**

**IDENTIFICAÇÃO:**
Doente do sexo feminino, 50 anos, professora, natural e residente em Leiria, solteira.

**DATA E MOTIVO DA ADMISSÃO:**
Admitida em 13/02/2025 por dor torácica atípica e fadiga progressiva.

**HISTÓRIA DA DOENÇA ATUAL:**
Paciente refere fadiga progressiva há cerca de 4 meses, acompanhada de episódios de dor torácica difusa e mal caracterizada, sem relação com esforço. Relata rigidez matinal nas articulações das mãos e joelhos, com duração superior a 1 hora. Nega febre, dispneia ou edema de membros inferiores.

**ANTECEDENTES PESSOAIS:**
- **Hipotiroidismo autoimune**, em tratamento com levotiroxina.
- **Nega tabagismo, etilismo ou outras comorbidades.**

**ANTECEDENTES FAMILIARES:**
- Mãe com artrite reumatoide.

**EXAME FÍSICO:**
- **Geral:** Paciente normotensa, eutrófica, sem sinais de sofrimento agudo.
- **Sinais Vitais:**
  - **FC:** 78 bpm
  - **PA:** 120/80 mmHg
  - **FR:** 16 irpm
  - **SpO₂:** 98% em ar ambiente
- **Músculo-esquelético:** Edema e dor à palpação em articulações metacarpofalângicas e interfalângicas proximais. Teste de compressão articular positivo.

**EXAMES COMPLEMENTARES:**
- **FAN:** Positivo (1:320, padrão pontilhado fino).
- **FR:** 75 UI/mL (elevado).
- **Anti-CCP:** 120 UI/mL (positivo).
- **PCR:** 12 mg/L (elevado).

**LISTA DE PROBLEMAS:**
1. Poliartrite inflamatória crónica.
2. Suspeita de artrite reumatoide.
3. Hipotiroidismo autoimune.

**PLANO:**
- Encaminhamento para reumatologia.
- Início de metotrexato com suplementação de ácido fólico.
- Monitorização laboratorial.
- Encaminhamento para fisioterapia.


"""





### Run the Model

**In the cell below, run the model. After a few seconds, the results will be displayed and saved in JSON format.**

# **Latest_model | Date: 18/05/2026**

In [ ]:
output=glintt_finetune(text=text1, model_id="tamunna/hfpt-glintt-balance", max_len=512)
print(f"--------------MODEL OUTPUT (text 1) -----------------")
print(output)

output=glintt_finetune(text=text2, model_id="tamunna/hfpt-glintt-balance", max_len=512)
print(f"--------------MODEL OUTPUT (text 2) -----------------")
print(output)



 cleaned text: NOTA CLÍNICA DE ADMISSÃO IDENTIFICAÇÃO: Doente do sexo masculino, 65 anos, aposentado, natural e residente em Coimbra, casado. DATA E MOTIVO DA ADMISSÃO: Admitido em 11/02/2025 por dispneia progressiva e edema de membros inferiores. HISTÓRIA DA DOENÇA ATUAL: Paciente refere início de dispneia há aproximadamente 3 meses, inicialmente aos grandes esforços, evoluindo progressivamente para pequenos esforços e, nos últimos dias, apresentando dispneia em repouso. Relata ortopneia e episódios de dispneia paroxística noturna. Associadamente, refere edema de membros inferiores, progressivo, bilateral, depressível, com piora ao final do dia. Nega dor torácica, palpitações ou síncope. ANTECEDENTES PESSOAIS: - Hipertensão arterial sistémica diagnosticada há 18 anos, em tratamento com enalapril. - Diabetes mellitus tipo 2, controlada com metformina. - Dislipidemia, em uso de sinvastatina. - Doença renal crónica grau III. - Ex - tabagista( 30 maços/ano) , cessou há 10 anos. ANTECEDE

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

--------------MODEL OUTPUT (text 1) -----------------
[
  {
    "p_label": "Medicação Habitual",
    "p_polaridade": null,
    "p_begin": 783,
    "p_end": 792,
    "p_word": "enalapril"
  },
  {
    "p_label": "Medicação Habitual",
    "p_polaridade": null,
    "p_begin": 837,
    "p_end": 847,
    "p_word": "metformina"
  },
  {
    "p_label": "Medicação Habitual",
    "p_polaridade": null,
    "p_begin": 875,
    "p_end": 887,
    "p_word": "sinvastatina"
  }
]


 cleaned text: NOTA CLÍNICA DE ADMISSÃO IDENTIFICAÇÃO: Doente do sexo feminino, 50 anos, professora, natural e residente em Leiria, solteira. DATA E MOTIVO DA ADMISSÃO: Admitida em 13/02/2025 por dor torácica atípica e fadiga progressiva. HISTÓRIA DA DOENÇA ATUAL: Paciente refere fadiga progressiva há cerca de 4 meses, acompanhada de episódios de dor torácica difusa e mal caracterizada, sem relação com esforço. Relata rigidez matinal nas articulações das mãos e joelhos, com duração superior a 1 hora. Nega febre, dispneia ou

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

--------------MODEL OUTPUT (text 2) -----------------
[
  {
    "p_label": "Medicação Habitual",
    "p_polaridade": null,
    "p_begin": 616,
    "p_end": 625,
    "p_word": "otiroxina"
  },
  {
    "p_label": "Diagnóstico",
    "p_polaridade": null,
    "p_begin": 1250,
    "p_end": 1282,
    "p_word": "Poliartrite inflamatória crónica"
  }
]
